In [1]:
# =========================
# Cell 1 — Imports & Global Config (fresh restart)
# =========================

# (Optional) pretty console
try:
    from rich import print as rprint
    from rich.console import Console
    from rich.table import Table
    from rich.panel import Panel
    console = Console()
except Exception:
    console = None
    rprint = print

import os, re, glob, math, random, json
import numpy as np
import pandas as pd

# Video IO
import cv2
cv2.setNumThreads(0)  # safer on Kaggle

# Storage
import h5py

# Torch
import torch
from torch import nn

# ---- Paths (dataset already provided on Kaggle) ----
BASE_DIR = "/kaggle/input/video-df/data"
ROB_DIR  = f"{BASE_DIR}/1"     # label=1 (robbery)
NON_DIR  = f"{BASE_DIR}/0"     # label=0 (non-robbery)

# ---- Output paths (we will rebuild everything cleanly) ----
OUT_DIR       = "/kaggle/working"
OUT_H5        = os.path.join(OUT_DIR, "robbery_frames_224.h5")
MANIFEST_CSV  = os.path.join(OUT_DIR, "manifest.csv")
SPLIT_NPZ     = os.path.join(OUT_DIR, "split_indices.npz")  # will be overwritten by later split cells

os.makedirs(OUT_DIR, exist_ok=True)

# ---- Packing / sampling parameters ----
MAX_FRAMES       = 80            # fixed T per video in the HDF5
RESIZE_HW        = (224, 224)    # (H, W) saved in HDF5
SAMPLE_STRATEGY  = "adaptive"    # "adaptive" (prefer) or "fixed"
SAMPLE_EVERY     = 10            # only used if SAMPLE_STRATEGY == "fixed"
MIN_REAL_FRAMES  = 8             # if fewer real frames → mark unusable
REPEAT_LAST_PAD  = False         # if True, pad by repeating last real frame; else zeros

# ---- HDF5 writer options ----
H5_COMPRESSION   = "gzip"
H5_COMP_OPTS     = 4
H5_SHUFFLE       = True
H5_FLETCHER32    = True          # integrity checksum on chunks

# ---- Metadata toggles (baked-in anti-leak keys) ----
WRITE_FAMILY_ID  = True          # e.g., "shop_lifter_210_1" → "shop_lifter_210"
WRITE_DOMAIN     = True          # e.g., prefix token like "shop_lifter" for domain splits

# ---- Training/eval (pretrained) defaults (used later) ----
CLIP_LEN         = 32            # r2plus1d_18 expects 32-frame clips
EVAL_CLIPS       = 5             # multi-clip eval per video
INPUT_SIZE_PT    = (3, 32, 112, 112)   # model input (C,T,H,W)
KINETICS_MEAN    = (0.43216, 0.394666, 0.37645)
KINETICS_STD     = (0.22803, 0.22145, 0.216989)

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

# ---- CUDA allocation hint to reduce fragmentation (helps T4) ----
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# ---- Small helpers used later in packing/splitting ----
_fam_regex = re.compile(r"^(.*?_\d+)_\d+$")
def family_of(vid: str) -> str:
    m = _fam_regex.match(vid)
    return m.group(1) if m else vid

def domain_of(vid: str) -> str:
    # Take alphabetic/underscore prefix before first number block as a simple domain
    m = re.match(r"^([a-zA-Z_]+)", vid)
    return m.group(1) if m else vid.split("_")[0]

# ---- Environment summary ----
num_devices = torch.cuda.device_count()
device_names = [torch.cuda.get_device_name(i) for i in range(num_devices)] if num_devices else []
versions = {
    "Python": f"{os.sys.version_info.major}.{os.sys.version_info.minor}.{os.sys.version_info.micro}",
    "PyTorch": torch.__version__,
    "TorchVision": None
}
try:
    import torchvision
    versions["TorchVision"] = torchvision.__version__
except Exception:
    versions["TorchVision"] = "N/A"

if console:
    # GPU table
    if num_devices == 0:
        console.print(Panel.fit("[red]No CUDA device detected.[/red]"))
    else:
        tbl = Table(title="CUDA Devices")
        tbl.add_column("#"); tbl.add_column("Name")
        for i, n in enumerate(device_names):
            tbl.add_row(str(i), n)
        console.print(tbl)

    # Versions table
    tv = Table(title="Library Versions")
    tv.add_column("Lib"); tv.add_column("Version")
    for k, v in versions.items():
        tv.add_row(k, v)
    # numpy / pandas / h5py versions
    import numpy, pandas, h5py as _h5py
    tv.add_row("NumPy", numpy.__version__)
    tv.add_row("Pandas", pandas.__version__)
    tv.add_row("h5py", _h5py.__version__)
    console.print(tv)

    # Config panel
    console.print(Panel.fit(
        f"[bold]Paths[/bold]\n"
        f"ROB_DIR : {ROB_DIR}\n"
        f"NON_DIR : {NON_DIR}\n"
        f"OUT_H5  : {OUT_H5}\n"
        f"MANIFEST: {MANIFEST_CSV}\n\n"
        f"[bold]Packing[/bold]\n"
        f"Strategy     : {SAMPLE_STRATEGY} (SAMPLE_EVERY={SAMPLE_EVERY})\n"
        f"MAX_FRAMES   : {MAX_FRAMES}\n"
        f"RESIZE_HW    : {RESIZE_HW}\n"
        f"MIN_REAL_FR  : {MIN_REAL_FRAMES}\n"
        f"Pad mode     : {'repeat-last' if REPEAT_LAST_PAD else 'zeros'}\n\n"
        f"[bold]HDF5[/bold]\n"
        f"compression  : {H5_COMPRESSION}({H5_COMP_OPTS}), shuffle={H5_SHUFFLE}, fletcher32={H5_FLETCHER32}\n"
        f"meta: family_id={WRITE_FAMILY_ID}, domain={WRITE_DOMAIN}\n\n"
        f"[bold]Pretrained (later)[/bold]\n"
        f"Clip len     : {CLIP_LEN}\n"
        f"Eval clips   : {EVAL_CLIPS}\n"
        f"Input size   : {INPUT_SIZE_PT}\n"
        f"Norm (mean)  : {KINETICS_MEAN}\n"
        f"Norm (std)   : {KINETICS_STD}\n"
    ))
else:
    print("CUDA devices:", device_names)
    print("Versions:", versions)
    print("OUT_H5:", OUT_H5)


  CUDA Devices  
┏━━━┳━━━━━━━━━━┓
┃ # ┃ Name     ┃
┡━━━╇━━━━━━━━━━┩
│ 0 │ Tesla T4 │
│ 1 │ Tesla T4 │
└───┴──────────┘

       Library Versions       
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Lib         ┃ Version      ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ Python      │ 3.11.13      │
│ PyTorch     │ 2.6.0+cu124  │
│ TorchVision │ 0.21.0+cu124 │
│ NumPy       │ 1.26.4       │
│ Pandas      │ 2.2.3        │
│ h5py        │ 3.14.0       │
└─────────────┴──────────────┘

╭───────────────────────────────────────────────────────╮
│ Paths                                                 │
│ ROB_DIR : /kaggle/input/video-df/data/1               │
│ NON_DIR : /kaggle/input/video-df/data/0               │
│ OUT_H5  : /kaggle/working/robbery_frames_224.h5       │
│ MANIFEST: /kaggle/working/manifest.csv                │
│                                                       │
│ Packing                                               │
│ Strategy     : adaptive (SAMPLE_EVERY=10)             │
│ MAX_FRAMES   : 80                                     │
│ RESIZE_HW    : (224, 224)                             │
│ MIN_REAL_FR  : 8                                      │
│ Pad mode     : zeros                                  │
│                                                       │
│ HDF5                                                  │
│ compression  : gzip(4), shuffle=True, fletcher32=True │
│ meta: family_id=True, domain=True                     │
│                                                       │
│ Pretrained (later)                                    │
│ Clip len     : 32                                     │
│ Eval clips   : 5                                      │
│ Input size   : (3, 32, 112, 112)                      │
│ Norm (mean)  : (0.43216, 0.394666, 0.37645)           │
│ Norm (std)   : (0.22803, 0.22145, 0.216989)           │
│                                                       │
╰───────────────────────────────────────────────────────╯

In [2]:
# =========================
# Cell 2 — Build Manifest (with family/domain metadata)
# =========================
import os, glob, re, pandas as pd

# Reuse helpers from Cell 1: family_of(), domain_of()
assert 'ROB_DIR' in globals() and 'NON_DIR' in globals() and 'MANIFEST_CSV' in globals()

# --- Enumerate video files (mp4 only; change pattern if needed) ---
rob_paths = sorted(glob.glob(os.path.join(ROB_DIR, "*.mp4")))
non_paths = sorted(glob.glob(os.path.join(NON_DIR, "*.mp4")))

rows = []
for p in rob_paths:
    vid = os.path.splitext(os.path.basename(p))[0]
    fam = family_of(vid) if WRITE_FAMILY_ID else vid
    dom = domain_of(vid)  if WRITE_DOMAIN    else ""
    rows.append({"video_id": vid, "path": p, "label": 1, "family_id": fam, "domain": dom})

for p in non_paths:
    vid = os.path.splitext(os.path.basename(p))[0]
    fam = family_of(vid) if WRITE_FAMILY_ID else vid
    dom = domain_of(vid)  if WRITE_DOMAIN    else ""
    rows.append({"video_id": vid, "path": p, "label": 0, "family_id": fam, "domain": dom})

manifest = pd.DataFrame(rows, columns=["video_id","path","label","family_id","domain"])

# --- Basic validations ---
dups_vid = manifest['video_id'].duplicated(keep=False).sum()
dups_path = manifest['path'].duplicated(keep=False).sum()
cls_counts = manifest['label'].value_counts().to_dict()

# --- Save ---
manifest.to_csv(MANIFEST_CSV, index=False)

# --- Pretty print summary ---
if 'console' in globals() and console:
    from rich.table import Table
    from rich.panel import Panel

    tbl = Table(title="Manifest Summary")
    tbl.add_column("Item"); tbl.add_column("Value")
    tbl.add_row("Robbery videos (label=1)", str(cls_counts.get(1,0)))
    tbl.add_row("Non-robbery videos (label=0)", str(cls_counts.get(0,0)))
    tbl.add_row("Total", str(len(manifest)))
    tbl.add_row("Duplicate video_id rows", str(int(dups_vid)))
    tbl.add_row("Duplicate path rows", str(int(dups_path)))
    tbl.add_row("Saved CSV", MANIFEST_CSV)
    console.print(tbl)

    # show first few rows
    console.print(Panel.fit(manifest.head(8).to_string(index=False), title="Manifest (head)"))
else:
    print("Saved manifest ->", MANIFEST_CSV)
    print("Counts:", cls_counts, "| dups(video_id)=", dups_vid, "dups(path)=", dups_path)
    print(manifest.head(8))


                       Manifest Summary                        
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Item                         ┃ Value                        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Robbery videos (label=1)     │ 0                            │
│ Non-robbery videos (label=0) │ 0                            │
│ Total                        │ 0                            │
│ Duplicate video_id rows      │ 0                            │
│ Duplicate path rows          │ 0                            │
│ Saved CSV                    │ /kaggle/working/manifest.csv │
└──────────────────────────────┴──────────────────────────────┘

╭─ Manifest (head) ─╮
│ Empty DataFrame   │
│ Columns:          │
│ Index: []         │
╰───────────────────╯

In [3]:
# =========================
# Cell 2 (retry) — Auto-discover dataset & rebuild manifest (robust, recursive)
# =========================
import os, re, glob, pandas as pd
from collections import defaultdict

# We rely on helpers from Cell 1:
# - family_of(vid)  -> grouping key
# - domain_of(vid)  -> simple domain prefix
# And globals: ROB_DIR, NON_DIR, MANIFEST_CSV, WRITE_FAMILY_ID, WRITE_DOMAIN

# ---- Config ----
EXTS = (".mp4", ".m4v", ".mov", ".avi", ".mkv", ".webm")

def count_media(path):
    if not os.path.isdir(path): return 0
    n = 0
    for root, _, files in os.walk(path):
        for f in files:
            if f.lower().endswith(EXTS):
                n += 1
    return n

def list_media(path):
    files = []
    for root, _, fnames in os.walk(path):
        for f in fnames:
            if f.lower().endswith(EXTS):
                files.append(os.path.join(root, f))
    return sorted(files)

def find_candidate_pairs():
    """
    Search /kaggle/input recursively for directories that contain a '0' and '1' subfolder with media.
    Return list of tuples: (rob_dir, non_dir, rob_count, non_count)
    """
    candidates = []
    scan_roots = [
        "/kaggle/input/video-df",           # common name
        "/kaggle/input",
    ]
    seen = set()
    for root in scan_roots:
        if not os.path.isdir(root): continue
        for cur, dirs, _ in os.walk(root):
            # quick prune: ignore hidden or very deep dirs
            if any(part.startswith('.') for part in cur.split(os.sep)): 
                continue
            # Look for sibling subdirs named '0' and '1'
            if "0" in dirs and "1" in dirs:
                nd = os.path.join(cur, "0")
                rd = os.path.join(cur, "1")
                key = (os.path.realpath(rd), os.path.realpath(nd))
                if key in seen: 
                    continue
                seen.add(key)
                c_r = count_media(rd)
                c_n = count_media(nd)
                if c_r + c_n > 0:
                    candidates.append((rd, nd, c_r, c_n))
    # sort by total descending
    candidates.sort(key=lambda x: x[2]+x[3], reverse=True)
    return candidates

def print_manifest(manifest: pd.DataFrame):
    from rich.table import Table
    from rich.panel import Panel
    tbl = Table(title="Manifest Summary")
    tbl.add_column("Item"); tbl.add_column("Value")
    cls_counts = manifest['label'].value_counts().to_dict() if len(manifest) else {}
    tbl.add_row("Robbery videos (label=1)", str(cls_counts.get(1,0)))
    tbl.add_row("Non-robbery videos (label=0)", str(cls_counts.get(0,0)))
    tbl.add_row("Total", str(len(manifest)))
    tbl.add_row("Saved CSV", MANIFEST_CSV)
    console.print(tbl)
    head = manifest.head(10).to_string(index=False) if len(manifest) else "EMPTY"
    console.print(Panel.fit(head, title="Manifest (head)"))

# ---- 1) Try the configured paths first ----
rob_count = count_media(ROB_DIR)
non_count = count_media(NON_DIR)

if rob_count == 0 or non_count == 0:
    console.print("[yellow]Configured paths yielded no videos. Searching /kaggle/input for class folders '1' and '0'...[/yellow]")
    pairs = find_candidate_pairs()
    if len(pairs) == 0:
        console.print("[red]No candidate (1/0) class folders with media were found under /kaggle/input.[/red]\n"
                      "Please verify the dataset is attached, or share the exact path.")
        # Save empty manifest (for reproducibility)
        pd.DataFrame(columns=["video_id","path","label","family_id","domain"]).to_csv(MANIFEST_CSV, index=False)
        print_manifest(pd.DataFrame(columns=["video_id","path","label","family_id","domain"]))
    else:
        # Pick the largest total
        new_rob, new_non, c_r, c_n = pairs[0]
        console.print(Panel.fit(
            f"[green]Using discovered dataset[/green]\n"
            f"ROB_DIR: {new_rob} (files={c_r})\n"
            f"NON_DIR: {new_non} (files={c_n})"
        ))
        ROB_DIR = new_rob
        NON_DIR = new_non
        rob_count, non_count = c_r, c_n

# ---- 2) Build the manifest from the chosen ROB_DIR / NON_DIR ----
rows = []
if rob_count > 0:
    for p in list_media(ROB_DIR):
        vid = os.path.splitext(os.path.basename(p))[0]
        fam = family_of(vid) if WRITE_FAMILY_ID else vid
        dom = domain_of(vid)  if WRITE_DOMAIN    else ""
        rows.append({"video_id": vid, "path": p, "label": 1, "family_id": fam, "domain": dom})
if non_count > 0:
    for p in list_media(NON_DIR):
        vid = os.path.splitext(os.path.basename(p))[0]
        fam = family_of(vid) if WRITE_FAMILY_ID else vid
        dom = domain_of(vid)  if WRITE_DOMAIN    else ""
        rows.append({"video_id": vid, "path": p, "label": 0, "family_id": fam, "domain": dom})

manifest = pd.DataFrame(rows, columns=["video_id","path","label","family_id","domain"]).sort_values("video_id").reset_index(drop=True)
manifest.to_csv(MANIFEST_CSV, index=False)

# ---- 3) Report ----
print_manifest(manifest)

# Also echo which folders we used now
console.print(Panel.fit(f"ROB_DIR -> {ROB_DIR}\nNON_DIR -> {NON_DIR}"))


Configured paths yielded no videos. Searching /kaggle/input for class folders '1' and '0'...

No candidate (1/0) class folders with media were found under /kaggle/input.
Please verify the dataset is attached, or share the exact path.

                       Manifest Summary                        
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Item                         ┃ Value                        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Robbery videos (label=1)     │ 0                            │
│ Non-robbery videos (label=0) │ 0                            │
│ Total                        │ 0                            │
│ Saved CSV                    │ /kaggle/working/manifest.csv │
└──────────────────────────────┴──────────────────────────────┘

╭─ Manifest (head) ─╮
│ EMPTY             │
╰───────────────────╯

                       Manifest Summary                        
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Item                         ┃ Value                        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Robbery videos (label=1)     │ 0                            │
│ Non-robbery videos (label=0) │ 0                            │
│ Total                        │ 0                            │
│ Saved CSV                    │ /kaggle/working/manifest.csv │
└──────────────────────────────┴──────────────────────────────┘

╭─ Manifest (head) ─╮
│ EMPTY             │
╰───────────────────╯

╭──────────────────────────────────────────╮
│ ROB_DIR -> /kaggle/input/video-df/data/1 │
│ NON_DIR -> /kaggle/input/video-df/data/0 │
╰──────────────────────────────────────────╯

In [4]:
# =========================
# Cell 2 — Find data automatically (use existing H5 if present; else raw 0/1 folders)
# =========================
import os, re, glob, h5py, pandas as pd, numpy as np

assert 'console' in globals(), "Run Cell 1 first."

def h5_is_valid(p):
    try:
        with h5py.File(p, "r") as f:
            for k in ("frames","labels","video_ids"):
                if k not in f: return False
            shp = f["frames"].shape  # (N,T,H,W,C)
            if not (len(shp)==5 and shp[-1]==3): return False
            return True
    except Exception:
        return False

# ---- 1) Prefer an existing packed H5 under /kaggle/input (fast path) ----
candidate_h5 = []
# Common name used earlier in your runs
candidate_h5 += glob.glob("/kaggle/input/uniform-robbery-video/*.h5")
# Fallback: any .h5/.hdf5 anywhere under /kaggle/input
candidate_h5 += glob.glob("/kaggle/input/**/*.h5", recursive=True)
candidate_h5 += glob.glob("/kaggle/input/**/*.hdf5", recursive=True)
candidate_h5 = [p for p in candidate_h5 if h5_is_valid(p)]

# Prefer the one that looks like our packed file
preferred = [p for p in candidate_h5 if "robbery_frames_224" in os.path.basename(p)]
H5_PATH = preferred[0] if preferred else (candidate_h5[0] if candidate_h5 else None)

used_source = None
manifest = None

if H5_PATH is not None:
    # ---- Build manifest straight from H5 metadata ----
    with h5py.File(H5_PATH, "r") as f:
        N, T, H, W, C = f["frames"].shape
        labels = f["labels"][:].astype(int)
        video_ids = [x.decode() for x in f["video_ids"][:]]
    # family/domain helpers from Cell 1
    fams = [family_of(v) if 'family_of' in globals() else v for v in video_ids]
    doms = [domain_of(v)  if 'domain_of'  in globals() else v.split('_')[0] for v in video_ids]
    manifest = pd.DataFrame({
        "video_id": video_ids,
        "path":    [f"h5://{i}" for i in range(len(video_ids))],  # placeholder (raw path unknown)
        "label":   labels,
        "family_id": fams,
        "domain": doms
    })
    manifest.to_csv(MANIFEST_CSV, index=False)
    used_source = f"H5: {H5_PATH}"

else:
    # ---- 2) Try raw folders with class 0/1 anywhere under /kaggle/input ----
    def count_media(path):
        exts = (".mp4", ".m4v", ".mov", ".avi", ".mkv", ".webm")
        n = 0
        for root, _, files in os.walk(path):
            for f in files:
                if f.lower().endswith(exts): n += 1
        return n

    def list_media(path):
        exts = (".mp4", ".m4v", ".mov", ".avi", ".mkv", ".webm")
        files = []
        for root, _, fnames in os.walk(path):
            for f in fnames:
                if f.lower().endswith(exts):
                    files.append(os.path.join(root, f))
        return sorted(files)

    # discover sibling "1" and "0" folders
    candidates = []
    for cur, dirs, _ in os.walk("/kaggle/input"):
        if "1" in dirs and "0" in dirs:
            rd = os.path.join(cur, "1")
            nd = os.path.join(cur, "0")
            cr, cn = count_media(rd), count_media(nd)
            if cr+cn > 0:
                candidates.append((rd, nd, cr, cn))
    candidates.sort(key=lambda x: x[2]+x[3], reverse=True)

    if candidates:
        ROB_DIR, NON_DIR, cr, cn = candidates[0]
        rows = []
        for p in list_media(ROB_DIR):
            vid = os.path.splitext(os.path.basename(p))[0]
            fam = family_of(vid) if 'family_of' in globals() else vid
            dom = domain_of(vid)  if 'domain_of'  in globals() else vid.split('_')[0]
            rows.append({"video_id": vid, "path": p, "label": 1, "family_id": fam, "domain": dom})
        for p in list_media(NON_DIR):
            vid = os.path.splitext(os.path.basename(p))[0]
            fam = family_of(vid) if 'family_of' in globals() else vid
            dom = domain_of(vid)  if 'domain_of'  in globals() else vid.split('_')[0]
            rows.append({"video_id": vid, "path": p, "label": 0, "family_id": fam, "domain": dom})
        manifest = pd.DataFrame(rows).sort_values("video_id").reset_index(drop=True)
        manifest.to_csv(MANIFEST_CSV, index=False)
        used_source = f"RAW: ROB_DIR={ROB_DIR} | NON_DIR={NON_DIR}"
    else:
        used_source = "NONE"

# ---- Report ----
from rich.table import Table
from rich.panel import Panel

tbl = Table(title="Data Source")
tbl.add_column("Used"); tbl.add_column("Details")
tbl.add_row("Source", used_source)
console.print(tbl)

if manifest is not None and len(manifest):
    cls_counts = manifest['label'].value_counts().to_dict()
    t = Table(title="Manifest Summary")
    t.add_column("Item"); t.add_column("Value")
    t.add_row("Robbery (1)", str(cls_counts.get(1,0)))
    t.add_row("Non-robbery (0)", str(cls_counts.get(0,0)))
    t.add_row("Total", str(len(manifest)))
    t.add_row("Saved CSV", MANIFEST_CSV)
    if H5_PATH is not None:
        t.add_row("H5_PATH", H5_PATH)
    console.print(t)
    console.print(Panel.fit(manifest.head(8).to_string(index=False), title="Manifest (head)"))
else:
    console.print(Panel.fit(
        "[red]No data found.[/red]\n"
        "• If you already have the packed file, add it as an input and ensure it appears at\n"
        "  /kaggle/input/uniform-robbery-video/robbery_frames_224.h5\n"
        "  or point H5_PATH at the correct .h5.\n"
        "• Or attach the raw dataset with class folders '1' and '0' somewhere under /kaggle/input.",
        title="Action needed"
    ))


                               Data Source                                
┏━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Used   ┃ Details                                                       ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Source │ H5: /kaggle/input/uniform-robbery-video/robbery_frames_224.h5 │
└────────┴───────────────────────────────────────────────────────────────┘

                               Manifest Summary                                
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Item            ┃ Value                                                     ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Robbery (1)     │ 324                                                       │
│ Non-robbery (0) │ 531                                                       │
│ Total           │ 855                                                       │
│ Saved CSV       │ /kaggle/working/manifest.csv                              │
│ H5_PATH         │ /kaggle/input/uniform-robbery-video/robbery_frames_224.h5 │
└─────────────────┴───────────────────────────────────────────────────────────┘

╭───────────────────── Manifest (head) ──────────────────────╮
│        video_id   path  label       family_id       domain │
│   shop_lifter_0 h5://0      1   shop_lifter_0 shop_lifter_ │
│   shop_lifter_1 h5://1      1   shop_lifter_1 shop_lifter_ │
│  shop_lifter_10 h5://2      1  shop_lifter_10 shop_lifter_ │
│ shop_lifter_100 h5://3      1 shop_lifter_100 shop_lifter_ │
│ shop_lifter_101 h5://4      1 shop_lifter_101 shop_lifter_ │
│ shop_lifter_102 h5://5      1 shop_lifter_102 shop_lifter_ │
│ shop_lifter_103 h5://6      1 shop_lifter_103 shop_lifter_ │
│ shop_lifter_104 h5://7      1 shop_lifter_104 shop_lifter_ │
╰────────────────────────────────────────────────────────────╯

In [5]:
# ====== Leak-proof split (FIXED H5 HANDLE) — family + similarity clusters + audits ======
import os, re, h5py, numpy as np, pandas as pd
from collections import defaultdict
from rich.table import Table
from rich.panel import Panel

assert 'H5_PATH' in globals(), "H5_PATH not set. Run the data discovery cell first."
assert os.path.exists(H5_PATH), f"Not found: {H5_PATH}"

# --- helpers ---
_fam_regex = re.compile(r"^(.*?_\d+)_\d+$")
def family_of(vid: str) -> str:
    m = _fam_regex.match(vid)
    return m.group(1) if m else vid

def domain_of(vid: str) -> str:
    m = re.match(r"^([a-zA-Z_]+)", vid)
    return m.group(1) if m else vid.split("_")[0]

class DSU:
    def __init__(self, n): self.p=list(range(n)); self.r=[0]*n
    def find(self,x):
        while self.p[x]!=x:
            self.p[x]=self.p[self.p[x]]; x=self.p[x]
        return x
    def union(self,a,b):
        ra,rb=self.find(a),self.find(b)
        if ra==rb: return
        if self.r[ra]<self.r[rb]: self.p[ra]=rb
        elif self.r[ra]>self.r[rb]: self.p[rb]=ra
        else: self.p[rb]=ra; self.r[ra]+=1

# --- params ---
FP_HW     = 16        # fingerprint downscale H,W (smaller => faster)
SIM_THR   = 0.999     # union clusters with cosine >= threshold
VAL_FRAC  = 0.20
SEEDS     = list(range(42, 60))  # try several to balance val ratio

# --- open H5 and compute fingerprints while the file is OPEN ---
with h5py.File(H5_PATH, "r") as f:
    frames = f["frames"]                  # (N,T,H,W,3) float16
    labels = f["labels"][:].astype(int)   # (N,)
    video_ids = np.array([x.decode() for x in f["video_ids"][:]])
    used = f["frames_used"][:] if "frames_used" in f else np.full(frames.shape[0], frames.shape[1], dtype=np.int32)

    N, T, H, W, C = frames.shape

    # mean-frame -> resize to FP_HW -> L2-normalized vector
    import torch, torch.nn.functional as F
    fps_list = []
    for i in range(N):
        u = int(max(1, min(int(used[i]), T)))
        arr = frames[i, :u]                              # (u,H,W,3) float16
        m = arr.mean(axis=0, dtype=np.float32)           # (H,W,3)
        t = torch.from_numpy(m).permute(2,0,1).unsqueeze(0)  # (1,3,H,W)
        t = F.interpolate(t, size=(FP_HW, FP_HW), mode='bilinear', align_corners=False)
        v = t.flatten().float()                          # (3*FP_HW*FP_HW,)
        v = v / (v.norm()+1e-8)
        fps_list.append(v.numpy())
    fps = np.stack(fps_list, axis=0)                     # (N,D)

# --- build clusters: family unions + similarity unions ---
dsu = DSU(N)

# (a) union by family
fam_to_idx = defaultdict(list)
for i, vid in enumerate(video_ids):
    fam_to_idx[family_of(vid)].append(i)
for idxs in fam_to_idx.values():
    base = idxs[0]
    for j in idxs[1:]:
        dsu.union(base, j)

# (b) union by cosine similarity
S = fps @ fps.T
np.fill_diagonal(S, 1.0)
r, c = np.where(np.triu(S, k=1) >= SIM_THR)
for i, j in zip(r.tolist(), c.tolist()):
    dsu.union(i, j)

roots = np.array([dsu.find(i) for i in range(N)], dtype=np.int64)
root_to_gid, gid = {}, 0
groups = np.empty(N, dtype=np.int64)
for i, r in enumerate(roots):
    if r not in root_to_gid:
        root_to_gid[r] = gid; gid += 1
    groups[i] = root_to_gid[r]
num_groups = gid

# --- remove conflicting-label clusters globally ---
g2labs = defaultdict(set)
for i in range(N): g2labs[groups[i]].add(int(labels[i]))
conflict_groups = sorted([g for g,s in g2labs.items() if len(s)>1])
keep_mask = np.array([groups[i] not in set(conflict_groups) for i in range(N)], dtype=bool)
KEEP = np.where(keep_mask)[0]
labels_k = labels[KEEP]
groups_k = groups[KEEP]
video_ids_k = video_ids[KEEP]

# --- StratifiedGroupKFold; pick split whose val class ratio is closest to overall ---
from sklearn.model_selection import StratifiedGroupKFold
n_splits = max(2, int(round(1/VAL_FRAC)))
overall_pos = (labels_k==1).mean()
best = None; best_score = 1e9

for seed in SEEDS:
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    tr_local, va_local = None, None
    for tr, va in sgkf.split(np.zeros(len(KEEP)), labels_k, groups=groups_k):
        tr_local, va_local = tr, va
        break
    val_pos = labels_k[va_local].mean() if len(va_local)>0 else 0.0
    size_pen = abs(len(va_local)/len(KEEP) - VAL_FRAC)
    score = abs(val_pos - overall_pos) + 0.5*size_pen
    if score < best_score:
        best_score = score; best = (tr_local, va_local, seed)

tr_local, va_local, used_seed = best
TRAIN_IDX = KEEP[np.sort(tr_local)]
VAL_IDX   = KEEP[np.sort(va_local)]

# --- audits: group overlap & similarity Val→Train (should be low after clustering) ---
train_g = set(groups[TRAIN_IDX].tolist())
val_g   = set(groups[VAL_IDX].tolist())
overlap_g = train_g.intersection(val_g)

fps_tr = fps[TRAIN_IDX]
fps_va = fps[VAL_IDX]
best_sim = (fps_va @ fps_tr.T).max(axis=1) if len(TRAIN_IDX) and len(VAL_IDX) else np.array([])

def dist_str(idx):
    c0 = int((labels[idx]==0).sum()); c1 = int((labels[idx]==1).sum()); tot = max(1, c0+c1)
    return f"{c0} ({c0/tot:.3f})", f"{c1} ({c1/tot:.3f})"

# --- save indices & class weights (train-only) ---
OUT_SPLIT = "/kaggle/working/split_indices_leakproof.npz"
np.savez(OUT_SPLIT, train_idx=TRAIN_IDX, val_idx=VAL_IDX)

num_classes = 2
counts_train = np.bincount(labels[TRAIN_IDX], minlength=num_classes).astype(np.float32)
weights_train = (len(TRAIN_IDX) / (num_classes * counts_train))

# --- pretty report ---
tbl = Table(title="Leak-Proof Split Summary")
tbl.add_column("Item"); tbl.add_column("Value")
tbl.add_row("Videos (N)", str(N))
tbl.add_row("Clusters", str(num_groups))
tbl.add_row("Conflict clusters removed", str(len(conflict_groups)))
tbl.add_row("Stratified seed", str(used_seed))
tbl.add_row("Train samples", str(len(TRAIN_IDX)))
tbl.add_row("Val samples", str(len(VAL_IDX)))
tbl.add_row("Group overlap", "OK ✅ (0)" if len(overlap_g)==0 else f"LEAK ❌ ({len(overlap_g)})")
console.print(tbl)

t = Table(title="Label Distribution")
t.add_column("Split"); t.add_column("Class 0"); t.add_column("Class 1")
c0o = int((labels==0).sum()); c1o = int((labels==1).sum()); to = c0o+c1o
t.add_row("Overall", f"{c0o} ({c0o/to:.3f})", f"{c1o} ({c1o/to:.3f})")
c0t,c1t = dist_str(TRAIN_IDX); t.add_row("Train", c0t, c1t)
c0v,c1v = dist_str(VAL_IDX);   t.add_row("Val",   c0v, c1v)
console.print(t)

if best_sim.size:
    p95 = float(np.percentile(best_sim, 95))
    p99 = float(np.percentile(best_sim, 99))
    c_98 = int((best_sim >= 0.98).sum())
    c_99 = int((best_sim >= 0.99).sum())
    c_995= int((best_sim >= 0.995).sum())
    c_999= int((best_sim >= 0.999).sum())
    t2 = Table(title="Nearest-Neighbor Similarity  (Val → Train)")
    t2.add_column("Stat"); t2.add_column("Value")
    t2.add_row("Val videos (Nv)", str(len(VAL_IDX)))
    t2.add_row("Train videos (Nt)", str(len(TRAIN_IDX)))
    t2.add_row("Mean best-sim", f"{best_sim.mean():.4f}")
    t2.add_row("P95 best-sim",   f"{p95:.4f}")
    t2.add_row("P99 best-sim",   f"{p99:.4f}")
    t2.add_row("Count ≥ 0.98",   str(c_98))
    t2.add_row("Count ≥ 0.99",   str(c_99))
    t2.add_row("Count ≥ 0.995",  str(c_995))
    t2.add_row("Count ≥ 0.999",  str(c_999))
    console.print(t2)

tw = Table(title="Class Weights (Train-only)")
tw.add_column("Class"); tw.add_column("Count"); tw.add_column("Weight")
for c in range(num_classes):
    tw.add_row(str(c), str(int(counts_train[c])), f"{weights_train[c]:.6f}")
console.print(tw)

console.print(Panel.fit(
    f"[green]Saved indices[/green] -> {OUT_SPLIT}\n"
    f"Use these for DataLoaders to avoid leakage.",
    title="Leak-proof split ready"
))


        Leak-Proof Split Summary         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Item                      ┃ Value     ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ Videos (N)                │ 855       │
│ Clusters                  │ 82        │
│ Conflict clusters removed │ 2         │
│ Stratified seed           │ 48        │
│ Train samples             │ 385       │
│ Val samples               │ 96        │
│ Group overlap             │ OK ✅ (0) │
└───────────────────────────┴───────────┘

          Label Distribution           
┏━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Split   ┃ Class 0     ┃ Class 1     ┃
┡━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ Overall │ 531 (0.621) │ 324 (0.379) │
│ Train   │ 225 (0.584) │ 160 (0.416) │
│ Val     │ 67 (0.698)  │ 29 (0.302)  │
└─────────┴─────────────┴─────────────┘

 Nearest-Neighbor Similarity  
        (Val → Train)         
┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Stat              ┃ Value  ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ Val videos (Nv)   │ 96     │
│ Train videos (Nt) │ 385    │
│ Mean best-sim     │ 0.9979 │
│ P95 best-sim      │ 0.9987 │
│ P99 best-sim      │ 0.9988 │
│ Count ≥ 0.98      │ 96     │
│ Count ≥ 0.99      │ 96     │
│ Count ≥ 0.995     │ 94     │
│ Count ≥ 0.999     │ 0      │
└───────────────────┴────────┘

 Class Weights (Train-only) 
┏━━━━━━━┳━━━━━━━┳━━━━━━━━━━┓
┃ Class ┃ Count ┃ Weight   ┃
┡━━━━━━━╇━━━━━━━╇━━━━━━━━━━┩
│ 0     │ 225   │ 0.855556 │
│ 1     │ 160   │ 1.203125 │
└───────┴───────┴──────────┘

╭─────────────────── Leak-proof split ready ───────────────────╮
│ Saved indices -> /kaggle/working/split_indices_leakproof.npz │
│ Use these for DataLoaders to avoid leakage.                  │
╰──────────────────────────────────────────────────────────────╯

In [6]:
# =========================
# One-cell fix: define missing globals, load safest split, rebuild loaders, smoke test
# =========================
import os, glob, re, h5py, numpy as np, torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ---- Rich (pretty logs) ----
try:
    from rich.console import Console
    from rich.table import Table
    from rich.panel import Panel
    console = Console()
    use_rich = True
except Exception:
    use_rich = False

# ---- Paths & defaults ----
H5_PATH = "/kaggle/input/uniform-robbery-video/robbery_frames_224.h5"
RESIZE_HW = (112, 112)   # for r2plus1d_18
CLIP_LEN  = 32
BATCH_TR  = 8
BATCH_VA  = 8
NUM_WORKERS = 2
PIN_MEMORY  = True

# ---- Kinetics-400 normalization (channel-first broadcasting) ----
KINETICS_MEAN = torch.tensor([0.43216, 0.394666, 0.37645], dtype=torch.float32).view(3,1,1,1)
KINETICS_STD  = torch.tensor([0.22803, 0.22145, 0.216989], dtype=torch.float32).view(3,1,1,1)

# ---- Pick the "best" split indices available under /kaggle/working ----
def _pick_split_npz():
    # Priority order (most leak-proof first)
    preferred = [
        "split_indices_balanced_strict.npz",
        "split_indices_leakproof_strict.npz",
        "split_indices_leakproof.npz",
        "split_indices_grouped_unique_clean.npz",
        "split_indices_grouped_unique.npz",
        "split_indices_grouped.npz",
        "split_indices.npz",
    ]
    for name in preferred:
        p = os.path.join("/kaggle/working", name)
        if os.path.exists(p):
            return p
    # Fallback: newest split_indices*.npz by mtime
    cand = sorted(glob.glob("/kaggle/working/split_indices*.npz"), key=os.path.getmtime, reverse=True)
    return cand[0] if cand else None

SPLIT_NPZ = _pick_split_npz()
if SPLIT_NPZ is None:
    raise FileNotFoundError("No split_indices*.npz found in /kaggle/working. Please run the split cell first.")

npz = np.load(SPLIT_NPZ)
train_idx = npz["train_idx"]
val_idx   = npz["val_idx"]

# ---- Uniform sampler over 'used' frames ----
def _uniform_indices(used: int, clip_len: int, train: bool):
    used = max(1, used)
    base = np.linspace(0, used-1, num=clip_len)
    if train and used > clip_len:
        step = (used-1) / (clip_len-1)
        jitter = np.random.uniform(-0.5*step, 0.5*step, size=clip_len)
        base = np.clip(base + jitter, 0, used-1)
    return np.round(base).astype(np.int64)

# ---- Dataset that outputs [C,T,H,W] normalized for r2plus1d ----
class H5Robbery32f(Dataset):
    def __init__(self, h5_path, indices, train: bool):
        self.h5_path = h5_path
        self.indices = np.array(indices, dtype=np.int64)
        self.train = train
        self.file = None
        with h5py.File(self.h5_path, "r") as f:
            self.T = f["frames"].shape[1]

    def _ensure_open(self):
        if self.file is None:
            # independent handle per worker/process
            self.file = h5py.File(self.h5_path, "r")

    def __len__(self): return len(self.indices)

    def __getitem__(self, i):
        self._ensure_open()
        idx = int(self.indices[i])
        frames = self.file["frames"][idx]            # (T,224,224,3) float16 [0,1]
        used   = int(self.file["frames_used"][idx])
        y      = int(self.file["labels"][idx])
        vid    = self.file["video_ids"][idx].decode()

        used = max(1, min(used, frames.shape[0]))
        sel = _uniform_indices(used, CLIP_LEN, train=self.train)  # (32,)
        clip = torch.from_numpy(frames[sel]).permute(0,3,1,2).to(torch.float32)  # [T,3,224,224]
        # resize to 112x112 treating T as batch
        clip = F.interpolate(clip, size=RESIZE_HW, mode="bilinear", align_corners=False)  # [T,3,112,112]
        clip = clip.permute(1,0,2,3).contiguous()  # [3,32,112,112]
        clip = (clip - KINETICS_MEAN) / KINETICS_STD
        return clip, y, vid

    def __del__(self):
        try:
            if self.file is not None: self.file.close()
        except: pass

# ---- Build loaders ----
train_ds_fix = H5Robbery32f(H5_PATH, train_idx, train=True)
val_ds_fix   = H5Robbery32f(H5_PATH, val_idx,   train=False)

# small smoke loader (num_workers=0) to avoid worker-setup issues first
tmp_loader = DataLoader(train_ds_fix, batch_size=4, shuffle=True, num_workers=0)
xb, yb, vids = next(iter(tmp_loader))

# main loaders
train_loader = DataLoader(train_ds_fix, batch_size=BATCH_TR, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=False)
val_loader   = DataLoader(val_ds_fix,   batch_size=BATCH_VA, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=False)

# ---- Report ----
if use_rich:
    tbl = Table(title="PT Batch Check (norm fix + globals defined)")
    tbl.add_column("Item"); tbl.add_column("Value")
    tbl.add_row("Split file", os.path.basename(SPLIT_NPZ))
    tbl.add_row("Train/Val", f"{len(train_ds_fix)} / {len(val_ds_fix)}")
    tbl.add_row("x", f"{tuple(xb.shape)}  (B,3,32,112,112)")
    tbl.add_row("y[:4]", str(yb[:4].tolist()))
    tbl.add_row("examples", ", ".join(vids[:3]))
    console.print(tbl)

    console.print(Panel.fit(
        f"[bold]Loaders rebuilt[/bold]\n"
        f"Train: {len(train_ds_fix)} | Val: {len(val_ds_fix)}\n"
        f"BATCH_TR={BATCH_TR}, BATCH_VA={BATCH_VA}, NUM_WORKERS={NUM_WORKERS}",
        title="Ready for training"
    ))
else:
    print("Split:", os.path.basename(SPLIT_NPZ))
    print("Train/Val:", len(train_ds_fix), "/", len(val_ds_fix))
    print("x:", xb.shape, "y[:4]:", yb[:4].tolist(), "examples:", vids[:3])
    print("Loaders rebuilt. Ready for training.")


             PT Batch Check (norm fix + globals defined)              
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Item       ┃ Value                                                 ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Split file │ split_indices_leakproof.npz                           │
│ Train/Val  │ 385 / 96                                              │
│ x          │ (4, 3, 32, 112, 112)  (B,3,32,112,112)                │
│ y[:4]      │ [1, 0, 1, 1]                                          │
│ examples   │ videyyyyyyyyyss_8, shop_lifter_n_46_1, shop_lifter_65 │
└────────────┴───────────────────────────────────────────────────────┘

╭───────── Ready for training ──────────╮
│ Loaders rebuilt                       │
│ Train: 385 | Val: 96                  │
│ BATCH_TR=8, BATCH_VA=8, NUM_WORKERS=2 │
╰───────────────────────────────────────╯

In [7]:
# =========================
# SAFE-MODE: r2plus1d_18 training (no DP, num_workers=0, AMP, hard-stop)
# =========================
import os, time, json, math, numpy as np, pandas as pd, h5py, torch, torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from torchvision.models.video import r2plus1d_18, R2Plus1D_18_Weights

# ---------- Config ----------
H5_PATH     = "/kaggle/input/uniform-robbery-video/robbery_frames_224.h5"
SPLIT_NPZ   = "/kaggle/working/split_indices_leakproof.npz"  # use your latest leak-proof split npz
CLIP_LEN    = 32
RESIZE_HW   = (112, 112)
BATCH_TR    = 4             # smaller, safer
BATCH_VA    = 4
NUM_WORKERS = 0             # critical: avoid h5py + multiprocessing crashes
PIN_MEMORY  = False
SEED        = 42

EPOCHS       = 20
LR           = 3e-4
WEIGHT_DECAY = 1e-4
PATIENCE     = 4
HARD_STOP_1  = True
AMP          = True

torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

# ---------- Kinetics-400 normalization ----------
KINETICS_MEAN = torch.tensor([0.43216, 0.394666, 0.37645], dtype=torch.float32).view(3,1,1,1)
KINETICS_STD  = torch.tensor([0.22803, 0.22145, 0.216989], dtype=torch.float32).view(3,1,1,1)

# ---------- Dataset (opens H5 inside each worker/process when accessed) ----------
class H5Robbery32f(Dataset):
    def __init__(self, h5_path, indices, train: bool):
        self.h5_path = h5_path
        self.indices = np.array(indices, dtype=np.int64)
        self.train = train
        self.file = None
        with h5py.File(self.h5_path, "r") as f:
            self.T = f["frames"].shape[1]

    def _ensure_open(self):
        if self.file is None:
            self.file = h5py.File(self.h5_path, "r")

    def __len__(self): return len(self.indices)

    def __getitem__(self, i):
        self._ensure_open()
        idx = int(self.indices[i])
        frames = self.file["frames"][idx]            # (T,224,224,3) float16 [0,1]
        used   = int(self.file["frames_used"][idx])
        y      = int(self.file["labels"][idx])
        vid    = self.file["video_ids"][idx].decode()

        used = max(1, min(used, frames.shape[0]))
        # uniform 32 indices with slight jitter in train
        base = np.linspace(0, used-1, num=CLIP_LEN)
        if self.train and used > CLIP_LEN:
            step = (used-1)/(CLIP_LEN-1)
            base = np.clip(base + np.random.uniform(-0.5*step, 0.5*step, size=CLIP_LEN), 0, used-1)
        sel = np.round(base).astype(np.int64)

        clip = torch.from_numpy(frames[sel]).permute(0,3,1,2).to(torch.float32)           # [T,3,224,224]
        clip = F.interpolate(clip, size=RESIZE_HW, mode="bilinear", align_corners=False)  # [T,3,112,112]
        clip = clip.permute(1,0,2,3).contiguous()                                         # [3,32,112,112]
        clip = (clip - KINETICS_MEAN) / KINETICS_STD
        return clip, y, vid

    def __del__(self):
        try:
            if self.file is not None: self.file.close()
        except: pass

# ---------- Load split & class weights ----------
sp = np.load(SPLIT_NPZ)
TRAIN_IDX, VAL_IDX = sp["train_idx"], sp["val_idx"]

with h5py.File(H5_PATH, "r") as f:
    labels_all = f["labels"][:].astype(int)
num_classes = 2
counts_train = np.bincount(labels_all[TRAIN_IDX], minlength=num_classes).astype(np.float32)
weights_train = (len(TRAIN_IDX) / (num_classes * counts_train))
class_weights = torch.tensor(weights_train, dtype=torch.float32)

# ---------- DataLoaders (safe mode) ----------
train_ds = H5Robbery32f(H5_PATH, TRAIN_IDX, train=True)
val_ds   = H5Robbery32f(H5_PATH, VAL_IDX,   train=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_TR, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_VA, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=False)

# quick smoke
xb, yb, _ = next(iter(train_loader))
assert xb.shape[1:] == (3, CLIP_LEN, *RESIZE_HW), f"Bad input shape: {xb.shape}"

# ---------- Model (single-GPU) ----------
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
weights = R2Plus1D_18_Weights.KINETICS400_V1
model = r2plus1d_18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler    = GradScaler("cuda" if device.type=="cuda" else "cpu", enabled=AMP)

def batch_acc(logits, target):
    pred = logits.argmax(dim=1)
    return (pred==target).sum().item(), target.numel()

@torch.no_grad()
def epoch_eval(loader):
    model.eval()
    total_loss, cor, tot = 0.0, 0, 0
    tp=fp=fn=0
    for xb, yb, _ in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        with autocast(device_type="cuda", enabled=AMP):
            logits = model(xb)
            loss = criterion(logits, yb)
        total_loss += loss.item()
        c,t = batch_acc(logits, yb); cor+=c; tot+=t
        pred = logits.argmax(dim=1)
        tp += int(((pred==1)&(yb==1)).sum().item())
        fp += int(((pred==1)&(yb==0)).sum().item())
        fn += int(((pred==0)&(yb==1)).sum().item())
    val_loss = total_loss / max(1, len(loader))
    val_acc  = cor / max(1, tot)
    precision = tp / max(1, tp+fp)
    recall    = tp / max(1, tp+fn)
    f1 = 0.0 if (precision+recall)==0 else 2*precision*recall/(precision+recall)
    return val_loss, val_acc, f1

def epoch_train(loader):
    model.train()
    total_loss, cor, tot = 0.0, 0, 0
    optimizer.zero_grad(set_to_none=True)
    for xb, yb, _ in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        with autocast(device_type="cuda", enabled=AMP):
            logits = model(xb)
            loss = criterion(logits, yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item()
        c,t = batch_acc(logits.detach(), yb); cor+=c; tot+=t
    return total_loss / max(1,len(loader)), cor / max(1,tot)

# ---------- Train ----------
best_val = -1.0
no_improve = 0
HIST_CSV = "/kaggle/working/history_pretrained_safe.csv"
CKPT     = "/kaggle/working/best_pretrained_safe.pth"
history = {"epoch": [], "train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_f1": []}
t0 = time.time()

for epoch in range(1, EPOCHS+1):
    torch.cuda.empty_cache()
    tr_loss, tr_acc = epoch_train(train_loader)
    va_loss, va_acc, va_f1 = epoch_eval(val_loader)

    history["epoch"].append(epoch)
    history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)
    history["val_loss"].append(va_loss);   history["val_acc"].append(va_acc); history["val_f1"].append(va_f1)

    print(f"Epoch {epoch}/{EPOCHS} | tr_loss {tr_loss:.4f} acc {tr_acc:.4f} || va_loss {va_loss:.4f} acc {va_acc:.4f} f1 {va_f1:.4f}")

    if va_acc > best_val:
        best_val = va_acc; no_improve = 0
        state = {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_acc": va_acc, "val_loss": va_loss
        }
        torch.save(state, CKPT)
        print(f"[saved] {CKPT} (val_acc={va_acc:.4f})")
    else:
        no_improve += 1

    if HARD_STOP_1 and (tr_acc >= 1.0 or va_acc >= 1.0):
        print(f"[HARD STOP] {'val' if va_acc>=1.0 else 'train'} accuracy hit 1.0 at epoch {epoch}.")
        break
    if no_improve >= PATIENCE:
        print(f"[Early stop] no improvement for {PATIENCE} epochs. Best val_acc={best_val:.4f}.")
        break

pd.DataFrame(history).to_csv(HIST_CSV, index=False)
print(f"Done. Best val_acc={best_val:.4f}. History -> {HIST_CSV} | CKPT -> {CKPT} | Time {(time.time()-t0)/60:.1f} min")


Downloading: "https://download.pytorch.org/models/r2plus1d_18-91a641e6.pth" to /root/.cache/torch/hub/checkpoints/r2plus1d_18-91a641e6.pth
100%|██████████| 120M/120M [00:00<00:00, 204MB/s]


Epoch 1/20 | tr_loss 0.2995 acc 0.8753 || va_loss 0.0054 acc 1.0000 f1 1.0000
[saved] /kaggle/working/best_pretrained_safe.pth (val_acc=1.0000)
[HARD STOP] val accuracy hit 1.0 at epoch 1.
Done. Best val_acc=1.0000. History -> /kaggle/working/history_pretrained_safe.csv | CKPT -> /kaggle/working/best_pretrained_safe.pth | Time 2.1 min
